<a href="https://colab.research.google.com/github/mach-ag01/data-science-portfolio/blob/main/02_Obesity_Levels_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥗 Project 2 — Obesity Levels Prediction

## Guided Google Colab Machine Learning Project

This notebook predicts **obesity levels** from lifestyle, eating habits, physical activity, family history, and demographic information.

### Workflow
1. 🧰 Setup
2. 📥 Load the UCI dataset
3. 🔍 Inspect the data
4. 🧹 Clean missing values and duplicates
5. 📊 Explore the target and important variables
6. ⚙️ Prepare numerical and categorical features
7. ✂️ Create training and test sets
8. 🤖 Train Logistic Regression, Random Forest and Neural Network models
9. 🔁 Compare models with cross-validation
10. 🧪 Evaluate the best model
11. 🔍 Interpret the results
12. 💾 Save project outputs

> Run the notebook from top to bottom. Every explanation is in a separate text cell before the related code.

## 1. 🧰 Setup

### 🔎 What's happening
We install and import the libraries needed for data loading, cleaning, visualization, preprocessing, model training, and evaluation.

In [ ]:
!pip -q install ucimlrepo

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)

print("Environment ready!")

## 2. 📥 Load the dataset

### 🔎 What's happening
We download the UCI Estimation of Obesity Levels dataset. We then separate the predictor variables from the obesity-level target.

In [ ]:
obesity = fetch_ucirepo(id=544)

X = obesity.data.features.copy()
y = obesity.data.targets.copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

## 3. 🔍 Inspect the dataset

### 🔎 What's happening
Before changing anything, we inspect the column names, data types, dataset structure, and target classes. This helps us decide how the data should be prepared.

In [ ]:
print("FEATURE INFORMATION")
print("=" * 60)
X.info()

print("\nFEATURE COLUMNS")
print(list(X.columns))

print("\nTARGET INFORMATION")
print("=" * 60)
print(y.columns)

target_column = y.columns[0]

print("\nOBESITY CLASSES")
display(y[target_column].value_counts())

## 4. 💾 Preserve the original data

### 🔎 What's happening
We keep an untouched copy of the original dataset. All cleaning and transformation will be performed on separate working copies.

In [ ]:
raw_X = X.copy()
raw_y = y.copy()

print("Original dataset preserved.")

## 5. 🧹 Check data quality

### 🔎 What's happening
We check for missing values and duplicate rows. Removing duplicates and understanding missing data helps prevent poor-quality training data from affecting the models.

In [ ]:
print("Missing values per feature:")
display(X.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", X.duplicated().sum())

X = X.drop_duplicates().reset_index(drop=True)
y = y.loc[X.index].reset_index(drop=True)

print("\nShape after duplicate handling:", X.shape)

## 6. 🧼 Handle missing values

### 🔎 What's happening
We replace common missing-value markers with proper missing values. The preprocessing pipeline later handles numerical and categorical missing values safely.

In [ ]:
X = X.replace(["?", "NA", "N/A", ""], np.nan)

print("Missing values:")
display(X.isna().sum().sort_values(ascending=False))

## 7. 📊 Explore the obesity target

### 🔎 What's happening
We visualize the number of records in each obesity category. This tells us whether some classes are much more common than others, which can affect model evaluation.

In [ ]:
plt.figure(figsize=(11, 6))

order = y[target_column].value_counts().index

sns.countplot(
    data=y,
    x=target_column,
    order=order
)

plt.title("Distribution of Obesity Levels")
plt.xlabel("Obesity Level")
plt.ylabel("Number of Individuals")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 8. 📏 Create and explore BMI

### 🔎 What's happening
The dataset contains height and weight, so we calculate Body Mass Index (BMI). This gives us an interpretable feature that can help reveal relationships between body measurements and obesity categories.

In [ ]:
X["BMI"] = X["Weight"] / (X["Height"] ** 2)

display(X[["Height", "Weight", "BMI"]].head())

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=pd.concat([X[["BMI"]], y], axis=1),
    x=target_column,
    y="BMI"
)

plt.title("BMI Distribution by Obesity Level")
plt.xlabel("Obesity Level")
plt.ylabel("BMI")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 9. 🍽️ Explore selected lifestyle variables

### 🔎 What's happening
We compare important lifestyle variables with obesity categories. These visualizations help us understand patterns before asking machine-learning models to learn from the data.

In [ ]:
selected_features = ["Age", "FCVC", "NCP", "FAF", "TUE"]

available = [c for c in selected_features if c in X.columns]

for feature in available:
    plt.figure(figsize=(10, 5))
    temp = pd.concat([X[[feature]], y], axis=1)
    sns.boxplot(data=temp, x=target_column, y=feature)
    plt.title(f"{feature} by Obesity Level")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

## 10. 🧩 Identify numerical and categorical features

### 🔎 What's happening
Machine-learning algorithms require numerical input. We identify which columns are numerical and which are categorical so that each type can receive the correct preprocessing.

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

## 11. ✂️ Create training and test datasets

### 🔎 What's happening
We reserve 20% of the data for final testing. The remaining data is used for model training and cross-validation. Stratification keeps the obesity-class proportions similar in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y[target_column],
    test_size=0.20,
    stratify=y[target_column],
    random_state=RANDOM_STATE
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

## 12. ⚙️ Build the preprocessing pipeline

### 🔎 What's happening
Numerical features are imputed and standardized. Categorical features are imputed and converted into one-hot encoded columns. Using a pipeline prevents data leakage and applies the same transformations consistently.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created.")

## 13. 🤖 Define the machine-learning models

### 🔎 What's happening
We compare three different approaches: Logistic Regression as a baseline, Random Forest as a tree-based ensemble model, and a Neural Network for nonlinear relationships.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=400,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "Neural Network": Pipeline([
        ("preprocessor", preprocessor),
        ("model", MLPClassifier(
            hidden_layer_sizes=(128, 64),
            max_iter=1000,
            early_stopping=True,
            random_state=RANDOM_STATE
        ))
    ])
}

print("Models ready:")
for name in models:
    print("-", name)

## 14. 🔁 Compare models using cross-validation

### 🔎 What's happening
Instead of judging a model from only one split, we use 5-fold stratified cross-validation. We compare accuracy and macro-averaged precision, recall, and F1-score so every obesity class contributes equally to the evaluation.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

results = []

for name, model in models.items():
    print(f"Evaluating {name}...")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Macro Precision": scores["test_precision_macro"].mean(),
        "Macro Recall": scores["test_recall_macro"].mean(),
        "Macro F1": scores["test_f1_macro"].mean()
    })

results_df = pd.DataFrame(results).sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

display(results_df)

## 15. 📈 Visualize the model comparison

### 🔎 What's happening
We create simple charts for the evaluation metrics. This makes it easier to see which model performs most consistently across the different measures.

In [ ]:
metrics = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]

for metric in metrics:
    plt.figure(figsize=(8, 5))
    sns.barplot(data=results_df, x="Model", y=metric)
    plt.title(f"Model Comparison — {metric}")
    plt.xticks(rotation=15)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

## 16. 🏆 Select and train the best model

### 🔎 What's happening
We select the model with the strongest cross-validated Macro F1 score. Macro F1 is useful here because this is a multi-class problem and we want balanced performance across all obesity levels.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Selected model:", best_model_name)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("Best model trained successfully.")

## 17. 🧪 Evaluate the final model

### 🔎 What's happening
We now evaluate the selected model on the test set that was not used during training. This gives us an independent estimate of final model performance.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("MODEL:", best_model_name)
print("=" * 70)

print(classification_report(y_test, y_pred))

print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 4))

## 18. 🔲 Create a confusion matrix

### 🔎 What's happening
The confusion matrix shows which obesity categories the model predicts correctly and where it makes mistakes. This helps identify classes that are difficult to distinguish.

In [ ]:
from sklearn.metrics import confusion_matrix

labels = sorted(y_test.unique())

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 19. 🔍 Examine feature importance

### 🔎 What's happening
Random Forest can estimate how useful each transformed feature was for making predictions. We fit a dedicated Random Forest model and inspect the most influential features.

In [ ]:
rf_pipeline = models["Random Forest"]
rf_pipeline.fit(X_train, y_train)

rf = rf_pipeline.named_steps["model"]
prep = rf_pipeline.named_steps["preprocessor"]

feature_names = prep.get_feature_names_out()

importance = pd.Series(
    rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

display(importance.head(20).to_frame("Importance"))

plt.figure(figsize=(10, 8))
importance.head(20).sort_values().plot(kind="barh")
plt.title("Top 20 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 20. 💾 Save the project outputs

### 🔎 What's happening
We save the cleaned feature data, target values, model comparison table, and feature-importance results. These files can later be used for a report, GitHub repository, or portfolio.

In [ ]:
os.makedirs("project2_outputs", exist_ok=True)

X.to_csv(
    "project2_outputs/obesity_features_with_bmi.csv",
    index=False
)

pd.DataFrame({
    "ObesityLevel": y[target_column]
}).to_csv(
    "project2_outputs/obesity_target.csv",
    index=False
)

results_df.to_csv(
    "project2_outputs/model_comparison.csv",
    index=False
)

importance.to_csv(
    "project2_outputs/feature_importance.csv"
)

print("Project outputs saved successfully.")

## 21. 📝 Final project summary

### 🔎 What's happening
This final section prints the most important results from the project. Use these actual values when writing your portfolio report and conclusion.

In [ ]:
best_row = results_df.iloc[0]

print("=" * 70)
print("PROJECT 2 — OBESITY LEVELS PREDICTION")
print("=" * 70)

print("\nBest model:")
print(best_model_name)

print("\nCross-validation performance:")
print(f"Accuracy        : {best_row['Accuracy']:.4f}")
print(f"Macro Precision : {best_row['Macro Precision']:.4f}")
print(f"Macro Recall    : {best_row['Macro Recall']:.4f}")
print(f"Macro F1        : {best_row['Macro F1']:.4f}")

print("\nFinal test accuracy:")
print(f"{accuracy_score(y_test, y_pred):.4f}")

print("\nProject completed successfully.")

# 🏁 Project 2 Conclusion

Use your **actual results** from the notebook to complete the statements below:

- The best-performing model was **[MODEL NAME]**.
- Its cross-validated Macro F1 score was **[RESULT]**.
- Its final test accuracy was **[RESULT]**.
- The most influential features included **[TOP FEATURES]**.

### Portfolio interpretation
This project demonstrates a complete multi-class machine-learning workflow: data acquisition, cleaning, exploratory analysis, feature engineering, categorical encoding, preprocessing pipelines, cross-validation, model comparison, test evaluation, and feature interpretation.

> **Important:** This model is an educational data-science project and should not be used as a clinical or medical assessment tool.